In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
for ep in w.serving_endpoints.list():
    print(ep.name)

In [0]:
%pip install -U databricks-langchain langchain "langgraph==1.0.5" "langgraph-prebuilt==1.0.5"
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langgraph.prebuilt import create_react_agent

catalog = "credit_risk_fraud_detection"
FOUNDATION_MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

llm = ChatDatabricks(endpoint=FOUNDATION_MODEL_ENDPOINT, temperature=0.1)

toolkit = UCFunctionToolkit(
    function_names=[
        f"{catalog}.ml.get_loan_risk_score",
        f"{catalog}.ml.get_loan_risk_factors",
        f"{catalog}.ml.get_fraud_score",
        f"{catalog}.ml.get_portfolio_summary",
        f"{catalog}.ml.get_dq_summary",
    ]
)
tools = toolkit.tools

SYSTEM_PROMPT = (
    "You are a credit risk and fraud analyst assistant for a lending portfolio. "
    "You have tools to: (1) look up a loan's delinquency escalation risk score "
    "and its risk factors given a loan_account_id, (2) look up a transaction's "
    "fraud probability given a transaction_id, (3) retrieve overall portfolio "
    "KPIs (default rate, fraud incidence, NPA by persona), and (4) retrieve a "
    "summary of current data quality violations in the pipeline. "
    "Use whichever tools are relevant to the question asked -- call multiple "
    "tools if the question has multiple parts. Be concise and factual: only "
    "state what the tool data actually supports, never invent numbers."
)

agent_executor = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

In [0]:
# MAGIC %md ## Test with a real loan_account_id

In [0]:
sample_loan_id = spark.sql(f"""
    select loan_account_id from {catalog}.gold.delinquency_risk_scores
    order by escalation_probability desc limit 1
""").collect()[0]["loan_account_id"]

sample_txn_id = spark.sql(f"""
    select transaction_id from {catalog}.gold.fraud_predictions
    order by fraud_probability desc limit 1
""").collect()[0]["transaction_id"]

question = (
    f"Explain the risk on loan {sample_loan_id}. "
    f"Also assess transaction {sample_txn_id} for fraud risk. "
    f"Then give me our overall portfolio default rate and fraud incidence rate, "
    f"and summarize our current top data quality violations."
)

result = agent_executor.invoke({"messages": [("human", question)]})
print(result["messages"][-1].content)

In [0]:
#MAGIC model registery

In [0]:
agent_code = '''
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langgraph.prebuilt import create_react_agent
import mlflow

catalog = "credit_risk_fraud_detection"
FOUNDATION_MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

llm = ChatDatabricks(endpoint=FOUNDATION_MODEL_ENDPOINT, temperature=0.1)

toolkit = UCFunctionToolkit(
    function_names=[
        f"{catalog}.ml.get_loan_risk_score",
        f"{catalog}.ml.get_loan_risk_factors",
        f"{catalog}.ml.get_fraud_score",
        f"{catalog}.ml.get_portfolio_summary",
        f"{catalog}.ml.get_dq_summary",
    ]
)
tools = toolkit.tools

SYSTEM_PROMPT = (
    "You are a credit risk and fraud analyst assistant for a lending portfolio. "
    "You have tools to: (1) look up a loan is delinquency escalation risk score "
    "and its risk factors given a loan_account_id, (2) look up a transaction is "
    "fraud probability given a transaction_id, (3) retrieve overall portfolio "
    "KPIs (default rate, fraud incidence, NPA by persona), and (4) retrieve a "
    "summary of current data quality violations in the pipeline. "
    "Use whichever tools are relevant to the question asked -- call multiple "
    "tools if the question has multiple parts. Be concise and factual: only "
    "state what the tool data actually supports, never invent numbers."
)

agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

mlflow.models.set_model(agent)
'''

with open("agent_code.py", "w") as f:
    f.write(agent_code)

In [0]:
import mlflow
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

input_example = {"messages": [{"role": "user", "content": f"Explain the risk on loan {sample_loan_id}"}]}
output_example = {"messages": [{"role": "assistant", "content": "Example risk explanation text."}]}
signature = infer_signature(input_example, output_example)

with mlflow.start_run(run_name="credit_risk_explainer_agent") as run:
    mlflow.langchain.log_model(
        lc_model="agent_code.py",
        artifact_path="agent",
        registered_model_name=f"{catalog}.ml.credit_risk_explainer_agent",
        signature=signature,
        input_example=input_example,
    )
    run_id = run.info.run_id

print("Run ID:", run_id)